[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/moeabench/blob/main/examples/full_analysis.ipynb)

# MoeaBench Masterclass: Comprehensive Algorithm Analysis
----------------------------------------------------------

This notebook demonstrates the full capabilities of `moeabench` for algorithmic diagnostics, focusing on the comparison between **NSGA-II** and **NSGA-III** on a 4-objective problem (**DTLZ2**).

We will cover:
1. **Experimentation**: Launching and managing multi-run experiments.
2. **Physical Metrics (`mb.metrics`)**: Raw indicators like IGD+, GD+, and Hypervolume.
3. **Statistical Analysis (`mb.stats`)**: Aggregating results and calculating probabilities.
4. **Visual Diagnostics (`mb.view`)**: Parallel coordinates, convergence history, and precision CDFs.
5. **Stratification (`mb.stats.strata`)**: Taxonomy of solution castes and Earth Mover's Distance.
6. **Scientific Audit (`mb.diagnostics`)**: The FAIR biopsy and clinical Q-Scores.

In [ ]:
# Install moeabench from GitHub
!pip install --quiet git+https://github.com/monacofj/moeabench.git

In [ ]:
import mb_path
from moeabench import mb
import numpy as np
import plotly.io as pio

# Ensure plots work in the notebook
pio.renderers.default = "notebook"

print(f"MoeaBench version: {mb.system.version()}")

## 1. Setup & Experimentation

We use `DTLZ2` with $M=4$ objectives. This dimensionality is a "tipping point":
- **NSGA-II** (Crowding Distance) begins to lose resolution in diversity mapping.
- **NSGA-III** (Reference Points) is designed to maintain structure in high-dimensional spaces.

In [ ]:
# Problem Definition
mop = mb.mops.DTLZ2(M=4)

# Experiment 1: NSGA-II
exp_n2 = mb.experiment()
exp_n2.name = "NSGA-II"
exp_n2.mop = mop
exp_n2.moea = mb.moeas.NSGA2(population=92, generations=200)

# Experiment 2: NSGA-III
exp_n3 = mb.experiment()
exp_n3.name = "NSGA-III"
exp_n3.mop = mop
exp_n3.moea = mb.moeas.NSGA3(population=92, generations=200)

# Execute Multi-run (3 runs each for stability)
print("Running NSGA-II...")
exp_n2.run(repeat=3)

print("Running NSGA-III...")
exp_n3.run(repeat=3)

## 2. Physical Metrics & Statistics

We analyze the raw results using standard Pareto-compliant indicators.

In [ ]:
# IGD+ Analysis
igdp_n2 = mb.metrics.igdp(exp_n2)
igdp_n3 = mb.metrics.igdp(exp_n3)

print("NSGA-II IGD+ Summary:")
igdp_n2.report()

print("\nNSGA-III IGD+ Summary:")
igdp_n3.report()

# Performance Probability (Statistical dominance of N3 over N2)
prob = mb.stats.perf_probability(igdp_n3, igdp_n2)
print(f"\nProbability NSGA-III > NSGA-II (based on IGD+): {prob:.2%}")

## 3. Visual Analysis

Visualization is key for understanding 4D objective spaces.

In [ ]:
# Parallel Coordinates View (Structural Diversity)
print("Visualizing Objective Space (Final Pareto Fronts)...")
mb.view.topo_shape(exp_n2.front(), exp_n3.front(), 
                   title="NSGA-II vs NSGA-III (4D DTLZ2 Fronts)")

# Convergence History (HV Progress)
hv_n2 = mb.metrics.hv(exp_n2)
hv_n3 = mb.metrics.hv(exp_n3)
mb.view.perf_history(hv_n2, hv_n3, title="Convergence Stability (Hypervolume History)")

# Precision Analysis (Distance-to-GT CDF)
# Shows what percentage of the front has reached a certain epsilon-closeness.
mb.view.perf_spread(exp_n2, exp_n3, title="Quality Distribution (Closeness CDF)")

## 4. Stratification Analysis

We investigate the internal hierarchy of the final populations.

In [ ]:
# Stratification into Caste/Tiers
strata_n3 = mb.stats.strata(exp_n3.front())
print("NSGA-III Stratification (Tiers):")
for tier_name, count in strata_n3.counts().items():
    print(f" - {tier_name}: {count} solutions")

# Caste Visualization
mb.view.strat_caste(exp_n3.front(), title="NSGA-III Caste Distribution")

# EMD (Earth Mover's Distance) - Effort to move N2 to N3
dist = mb.stats.emd(exp_n2.front(), exp_n3.front())
print(f"\nGlobal Topographic Displacement (EMD): {dist:.4f}")

## 5. Clinical Audit (FAIR Diagnostics)

The final step is the scientific audit that interprets the results through the FAIR (Finite Approximation-Induced Resolution) framework.

In [ ]:
# Execute Clinical Audit
audit_n2 = mb.diagnostics.audit(exp_n2)
audit_n3 = mb.diagnostics.audit(exp_n3)

print("--- NSGA-II BIOPSY REPORT ---")
audit_n2.report()

print("\n--- NSGA-III BIOPSY REPORT ---")
audit_n3.report()

## 6. Synthesis

In this 4D scenario, observe how:
- **NSGA-III** typically shows a higher **Q_BALANCE** and **Q_COVERAGE** due to its reference points providing a uniform structural scaffold.
- **NSGA-II** might show excellent **Q_CLOSENESS**, but its **Q_BALANCE** may degrade as crowding distance struggles to differentiate between solutions in higher dimensions.

This diagnostic profile allows you to choose the algorithm not just based on "better IGD", but based on the specific **clinical failure modes** your problem can tolerate.